# 04b · Predicciones de M3 sobre el conjunto de prueba (CPU)

En la corrida de Kaggle (notebook `04`), el modelo se entrenó correctamente: la mejor configuración alcanzó una exactitud de 0.845 en validación en la época 4. Sin embargo, la celda de verificación de ese notebook mostró que las predicciones generadas al recargar el modelo **dentro de la sesión de Kaggle con GPU** eran degeneradas (exactitud de 0.58 en validación, prediciendo solo `True_Correct` o `False_Neither`).

Se comprobó que el archivo del modelo guardado (`models/m3_deberta/`) está bien: cargado en CPU con `src/prediccion.py` reproduce el desempeño del entrenamiento. Por eso, las predicciones sobre el conjunto de prueba se regeneran aquí, en CPU, con la misma función que usa la aplicación.

**Salidas:** `results/predicciones/pred_M3_deberta.csv` y la fila de M3 en `results/tiempos.csv` (el tiempo de entrenamiento es el de Kaggle; el de inferencia, el de esta corrida en CPU).

In [1]:
import sys
import time
from pathlib import Path

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import torch
import transformers

from src import preprocesamiento as pp
from src import prediccion
from src.metricas import resumen_metricas
from src.utilidades import guardar_predicciones, registrar_tiempo

NOMBRE = "M3_deberta"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU:", torch.cuda.is_available())
modelo = prediccion.cargar_modelo(NOMBRE)
{k: v for k, v in modelo.info.items() if k in ["modelo_base", "max_len", "usar_mascara", "hiperparametros", "metricas_validacion"]}

/tmp/claude-0/-home-claude/2e66f52c-1ac3-5d2a-8f00-7ffcacb867dd/scratchpad/t5/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.14.1+cu130 | transformers 5.19.0 | GPU: False


/tmp/claude-0/-home-claude/2e66f52c-1ac3-5d2a-8f00-7ffcacb867dd/scratchpad/venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(



Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 6431.81it/s]

{'modelo_base': 'microsoft/deberta-v3-base',
 'max_len': 128,
 'usar_mascara': True,
 'hiperparametros': {'learning_rate': 3e-05,
  'perdida_ponderada': True,
  'epocas_max': 4,
  'batch': 16,
  'warmup': 0.1,
  'weight_decay': 0.01},
 'metricas_validacion': {'f1_macro': 0.5832533404387313,
  'accuracy': 0.8454198473282443,
  'map_at_3': 0.9162577244638312}}

## 1. Verificación en validación

Debe reproducir las métricas de validación registradas durante el entrenamiento en Kaggle.

In [2]:
datos = pp.aplicar_particion(pp.cargar_train_clean())
pred_val = prediccion.predecir_lote(NOMBRE, datos["val"])
resumen_metricas(datos["val"]["target"], pred_val[["top1", "top2", "top3"]].values)

{'f1_macro': 0.5832533404387313,
 'accuracy': 0.8454198473282443,
 'map_at_3': 0.9162577244638312}

## 2. Predicciones sobre el conjunto de prueba

In [3]:
inicio = time.time()
pred_test = prediccion.predecir_lote(NOMBRE, datos["test"])
t_inferencia = time.time() - inicio

ruta_pred = guardar_predicciones(pred_test, NOMBRE)
t_entrenamiento = pd.read_csv(RAIZ / "results/tiempos.csv").set_index("modelo").loc[NOMBRE, "tiempo_entrenamiento_s"]
registrar_tiempo(NOMBRE, t_entrenamiento, t_inferencia, hardware="GPU Tesla T4 (entrenamiento) / CPU (inferencia)")
print(f"{len(pred_test):,} predicciones en {t_inferencia:.1f} s -> {ruta_pred.relative_to(RAIZ)}")
pred_test.head()

7,340 predicciones en 809.5 s -> results/predicciones/pred_M3_deberta.csv


,row_id,top1,top2,top3,prob1,prob2,prob3
0,0,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.604529,0.392751,0.001542
1,16,True_Neither:NA,True_Correct:NA,True_Misconception:Incomplete,0.935338,0.063359,0.000683
2,19,True_Correct:NA,True_Neither:NA,True_Misconception:WNB,0.892595,0.106099,0.000746
3,24,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.964432,0.035046,0.000267
4,26,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.891442,0.107542,0.000530


In [4]:
pd.read_csv(RAIZ / "results/tiempos.csv")

,modelo,tiempo_entrenamiento_s,tiempo_inferencia_test_s,hardware
0,M1_tfidf_lr,99.81,1.54,CPU (x86_64)
1,M3_deberta,1665.70,809.53,GPU Tesla T4 (entrenamiento) / CPU (inferencia)
